# Hospital Readmission RAG System

## RAG Workflow

1. Load authoritative AHRQ and CMS source documents.
2. Extract and clean the document text.
3. Divide the documents into smaller text chunks.
4. Create embeddings for each chunk.
5. Store the embeddings in a vector-based retrieval index.
6. Retrieve the most relevant guidance for a user's question.
7. Provide the retrieved evidence to an LLM.
8. Generate practical post-discharge recommendations with source and page references.

The RAG system is intended as a decision-support tool and does not replace clinical judgment.


## 1. Install and Import RAG Libraries

In [ ]:
!pip install -q --upgrade openai scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 105.7 MB/s eta 0:00:00


In [ ]:
import os
import re
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from openai import OpenAI

from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

SEED = 123

random.seed(SEED)
np.random.seed(SEED)

print("Imports loaded successfully.")

Imports loaded successfully.


## Configure the RAG System

This section defines the readmission target, patient features, retrieval settings, and OpenAI models used for embeddings and classification.

In [ ]:
LABEL_COL = "readmitted_30"

FEATURE_COLS = [
    "age",
    "time_in_hospital",
    "num_medications",
    "number_diagnoses",
    "number_inpatient",
    "number_emergency",
    "number_outpatient",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
    "A1Cresult",
    "insulin",
    "diabetesMed",
]

TOP_K = 25

# Start small so we can make sure the RAG pipeline works first
N_TO_CLASSIFY = 100

EMBEDDING_MODEL = "text-embedding-3-small"
CLASSIFICATION_MODEL = "gpt-5-nano"

print("Top K:", TOP_K)
print("Patients to classify:", N_TO_CLASSIFY)
print("Embedding model:", EMBEDDING_MODEL)
print("Classification model:", CLASSIFICATION_MODEL)

Top K: 25
Patients to classify: 100
Embedding model: text-embedding-3-small
Classification model: gpt-5-nano


In [ ]:
from google.colab import userdata

openai_api_key = userdata.get("OPENAI_API_KEY")

if not openai_api_key:
    raise RuntimeError(
        "OPENAI_API_KEY was not found in Colab Secrets."
    )

client = OpenAI(
    api_key=openai_api_key
)

print("OpenAI client initialized.")

OpenAI client initialized.


## Load the Cleaned Diabetes Dataset

In [ ]:
df = pd.read_csv("cleaned_diabetes_data.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (99235, 44)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_30,diag_1_group,diag_2_group,diag_3_group
0,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,41,0,...,No,No,No,No,No,No,0,Diabetes,Unknown,Unknown
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,59,0,...,No,No,No,No,Ch,Yes,0,Other,Diabetes,Other
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,11,5,...,No,No,No,No,No,Yes,0,Other,Diabetes,Other
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,44,1,...,No,No,No,No,Ch,Yes,0,Other,Diabetes,Circulatory
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,51,0,...,No,No,No,No,Ch,Yes,0,Neoplasms,Neoplasms,Diabetes


##  Validate and Prepare the Data

The RAG system uses the same patient predictors as the readmission models. The target variable is retained for evaluating predictions but is excluded from the patient text used for retrieval and classification.

In [ ]:
required_cols = FEATURE_COLS + [LABEL_COL]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise KeyError(
        f"Dataset is missing columns: {missing_cols}"
    )

rag_df = df[required_cols].copy()

rag_df[LABEL_COL] = pd.to_numeric(
    rag_df[LABEL_COL],
    errors="coerce"
)

rag_df = rag_df.dropna(
    subset=[LABEL_COL]
).copy()

rag_df[LABEL_COL] = (
    rag_df[LABEL_COL].astype(int)
)

print("Rows available:", len(rag_df))

print("\nReadmission distribution:")
print(
    rag_df[LABEL_COL]
    .value_counts()
    .sort_index()
)

print("\nReadmission proportions:")
print(
    rag_df[LABEL_COL]
    .value_counts(normalize=True)
    .sort_index()
)

Rows available: 99235

Readmission distribution:
readmitted_30
0    87936
1    11299
Name: count, dtype: int64

Readmission proportions:
readmitted_30
0    0.886139
1    0.113861
Name: proportion, dtype: float64


## Create Training and Test Sets

The dataset is split into training and held-out test sets using stratified sampling so the original readmission rate is maintained in both sets.

In [ ]:
train_df, test_df = train_test_split(
    rag_df,
    test_size=0.20,
    random_state=SEED,
    stratify=rag_df[LABEL_COL]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "\nTraining readmission rate:",
    round(train_df[LABEL_COL].mean() * 100, 2),
    "%"
)

print(
    "Testing readmission rate:",
    round(test_df[LABEL_COL].mean() * 100, 2),
    "%"
)

Training rows: 79388
Test rows: 19847

Training readmission rate: 11.39 %
Testing readmission rate: 11.39 %


## Convert patients to retrieval text

In [ ]:
def format_value(value):
    if pd.isna(value):
        return "Missing"

    if isinstance(value, (float, np.floating)):
        return (
            f"{float(value):.2f}"
            .rstrip("0")
            .rstrip(".")
        )

    return str(value)


def patient_to_text(row):
    return (
        f"age: {format_value(row['age'])}; "
        f"time in hospital: {format_value(row['time_in_hospital'])} days; "
        f"number of medications: {format_value(row['num_medications'])}; "
        f"number of diagnoses: {format_value(row['number_diagnoses'])}; "
        f"previous inpatient visits: {format_value(row['number_inpatient'])}; "
        f"previous emergency visits: {format_value(row['number_emergency'])}; "
        f"previous outpatient visits: {format_value(row['number_outpatient'])}; "
        f"primary diagnosis group: {format_value(row['diag_1_group'])}; "
        f"secondary diagnosis group: {format_value(row['diag_2_group'])}; "
        f"additional diagnosis group: {format_value(row['diag_3_group'])}; "
        f"A1C result: {format_value(row['A1Cresult'])}; "
        f"insulin: {format_value(row['insulin'])}; "
        f"diabetes medication: {format_value(row['diabetesMed'])}"
    )


train_df["retrieval_text"] = train_df.apply(
    patient_to_text,
    axis=1
)

test_df["retrieval_text"] = test_df.apply(
    patient_to_text,
    axis=1
)

print("Example patient retrieval text:\n")
print(train_df.loc[0, "retrieval_text"])

Example patient retrieval text:

age: [40-50); time in hospital: 2 days; number of medications: 29; number of diagnoses: 5; previous inpatient visits: 0; previous emergency visits: 0; previous outpatient visits: 1; primary diagnosis group: Diabetes; secondary diagnosis group: Circulatory; additional diagnosis group: Genitourinary; A1C result: Not Recorded; insulin: No; diabetes medication: No


## Create a Stratified Retrieval Sample

In [ ]:
# Creating a smaller retrieval set that keeps the original class distribution
retrieval_df, _ = train_test_split(
    train_df,
    train_size=10000,
    random_state=SEED,
    stratify=train_df[LABEL_COL]
)

retrieval_df = retrieval_df.reset_index(drop=True)

print("Retrieval rows:", len(retrieval_df))

print("\nReadmission distribution:")
print(
    retrieval_df[LABEL_COL]
    .value_counts()
    .sort_index()
)

print("\nReadmission proportions:")
print(
    retrieval_df[LABEL_COL]
    .value_counts(normalize=True)
    .sort_index()
)

Retrieval rows: 10000

Readmission distribution:
readmitted_30
0    8861
1    1139
Name: count, dtype: int64

Readmission proportions:
readmitted_30
0    0.8861
1    0.1139
Name: proportion, dtype: float64


## Create Embeddings and Build the Retrieval Index

In [ ]:
def embed_texts(texts, model, batch_size=250):
    # Creating embeddings in batches to reduce API load
    texts = [str(text) for text in texts]
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        response = client.embeddings.create(
            model=model,
            input=batch
        )

        batch_embeddings = [
            item.embedding
            for item in response.data
        ]

        all_embeddings.extend(
            batch_embeddings
        )

        print(
            f"Embedded "
            f"{min(start + batch_size, len(texts)):,} "
            f"of {len(texts):,} rows"
        )

    return np.array(
        all_embeddings,
        dtype=np.float32
    )


# Creating embeddings for the historical patient encounters
retrieval_embeddings = embed_texts(
    retrieval_df["retrieval_text"].tolist(),
    EMBEDDING_MODEL
)

print(
    "\nEmbedding matrix shape:",
    retrieval_embeddings.shape
)

Embedded 250 of 10,000 rows
Embedded 500 of 10,000 rows
Embedded 750 of 10,000 rows
Embedded 1,000 of 10,000 rows
Embedded 1,250 of 10,000 rows
Embedded 1,500 of 10,000 rows
Embedded 1,750 of 10,000 rows
Embedded 2,000 of 10,000 rows
Embedded 2,250 of 10,000 rows
Embedded 2,500 of 10,000 rows
Embedded 2,750 of 10,000 rows
Embedded 3,000 of 10,000 rows
Embedded 3,250 of 10,000 rows
Embedded 3,500 of 10,000 rows
Embedded 3,750 of 10,000 rows
Embedded 4,000 of 10,000 rows
Embedded 4,250 of 10,000 rows
Embedded 4,500 of 10,000 rows
Embedded 4,750 of 10,000 rows
Embedded 5,000 of 10,000 rows
Embedded 5,250 of 10,000 rows
Embedded 5,500 of 10,000 rows
Embedded 5,750 of 10,000 rows
Embedded 6,000 of 10,000 rows
Embedded 6,250 of 10,000 rows
Embedded 6,500 of 10,000 rows
Embedded 6,750 of 10,000 rows
Embedded 7,000 of 10,000 rows
Embedded 7,250 of 10,000 rows
Embedded 7,500 of 10,000 rows
Embedded 7,750 of 10,000 rows
Embedded 8,000 of 10,000 rows
Embedded 8,250 of 10,000 rows
Embedded 8,500 o

In [ ]:
# Building the nearest-neighbor retrieval index
neighbor_index = NearestNeighbors(
    n_neighbors=TOP_K,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

neighbor_index.fit(
    retrieval_embeddings
)

print(
    "Nearest-neighbor index ready with K =",
    TOP_K
)

Nearest-neighbor index ready with K = 25


## Create the Evidence-Based RAG Knowledge Base

The RAG knowledge base contains authoritative guidance from the Agency for Healthcare Research and Quality (AHRQ) and the Centers for Medicare & Medicaid Services (CMS).

These sources provide evidence-based information related to hospital discharge planning, care transitions, post-discharge follow-up, medication management, and reducing preventable readmissions.



In [ ]:
RAG_SOURCES = [
    {
        "source": "AHRQ",
        "title": "Re-Engineered Discharge (RED) Toolkit",
        "url": (
            "https://www.ahrq.gov/patient-safety/"
            "settings/hospital/red/toolkit/index.html"
        ),
    },
    {
        "source": "AHRQ",
        "title": "Postdischarge Follow-up Phone Call",
        "url": (
            "https://www.ahrq.gov/patient-safety/"
            "settings/hospital/red/toolkit/redtool5.html"
        ),
    },
    {
        "source": "CMS",
        "title": "Hospital Readmissions Reduction Program",
        "url": (
            "https://www.cms.gov/medicare/quality/"
            "value-based-programs/hospital-readmissions"
        ),
    },
]

sources_df = pd.DataFrame(RAG_SOURCES)

display(sources_df)

,source,title,url
0,AHRQ,Re-Engineered Discharge (RED) Toolkit,https://www.ahrq.gov/patient-safety/settings/h...
1,AHRQ,Postdischarge Follow-up Phone Call,https://www.ahrq.gov/patient-safety/settings/h...
2,CMS,Hospital Readmissions Reduction Program,https://www.cms.gov/medicare/quality/value-bas...


## Download and Clean the Source Documents

The text from the selected AHRQ and CMS sources is downloaded and cleaned before being divided into smaller chunks for retrieval.

In [ ]:
!pip install -q pypdf

In [ ]:
from pypdf import PdfReader

In [ ]:
SOURCE_FILES = [
    {
        "source": "AHRQ",
        "title": "Re-Engineered Discharge (RED) Toolkit",
        "file": "Re-Engineered Discharge (RED) Toolkit _ Agency for Healthcare Research and Quality.pdf",
    },
    {
        "source": "AHRQ",
        "title": "How To Conduct a Postdischarge Follow-up Phone Call",
        "file": "Tool 5_ How To Conduct a Postdischarge Followup Phone Call _ Agency for Healthcare Research and Quality.pdf",
    },
    {
        "source": "CMS",
        "title": "Hospital Readmissions Reduction Program",
        "file": "Hosp. Readmission Reduction _ CMS.pdf",
    },
]

In [ ]:
# Reading text from each PDF page
def read_pdf_text(file_path):
    reader = PdfReader(file_path)

    pages = []

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):
        text = page.extract_text()

        if text:
            # Cleaning extra spaces
            clean_text = " ".join(
                text.split()
            )

            pages.append({
                "page": page_number,
                "text": clean_text
            })

    return pages

In [ ]:
documents = []

for source in SOURCE_FILES:
    pages = read_pdf_text(
        source["file"]
    )

    for page in pages:
        documents.append({
            "source": source["source"],
            "title": source["title"],
            "page": page["page"],
            "text": page["text"]
        })

documents_df = pd.DataFrame(
    documents
)

print(
    "Pages loaded:",
    len(documents_df)
)

display(
    documents_df[
        [
            "source",
            "title",
            "page"
        ]
    ].head(10)
)

Pages loaded: 17


,source,title,page
0,AHRQ,Re-Engineered Discharge (RED) Toolkit,1
1,AHRQ,Re-Engineered Discharge (RED) Toolkit,2
2,AHRQ,Re-Engineered Discharge (RED) Toolkit,3
3,AHRQ,Re-Engineered Discharge (RED) Toolkit,4
4,AHRQ,Re-Engineered Discharge (RED) Toolkit,5
5,AHRQ,How To Conduct a Postdischarge Follow-up Phone...,1
6,AHRQ,How To Conduct a Postdischarge Follow-up Phone...,2
7,AHRQ,How To Conduct a Postdischarge Follow-up Phone...,3
8,AHRQ,How To Conduct a Postdischarge Follow-up Phone...,4
9,AHRQ,How To Conduct a Postdischarge Follow-up Phone...,5


In [ ]:
for title in documents_df["title"].unique():
    example_text = (
        documents_df[
            documents_df["title"] == title
        ]["text"].iloc[0]
    )

    print(title)
    print(example_text[:500])
    print("\n" + "-" * 80 + "\n")

Re-Engineered Discharge (RED) Toolkit
(https://www.ahrq.gov/) Re-Engineered Discharge (RED) Toolkit Next Page Table of Contents https://www.ahrq.gov/patient-safety/settings/hospital/red/toolkit/index.html A variety of forces are pushing hospitals to improve their discharge processes to reduce readmissions. Researchers at the Boston University Medical Center (BUMC) developed and tested the Re-Engineered Discharge (RED). Research showed that the RED was effective at reducing readmissions and posthospital emergency department (ED) visi

--------------------------------------------------------------------------------

How To Conduct a Postdischarge Follow-up Phone Call
(https://www.ahrq.gov/) Re-Engineered Discharge (RED) Toolkit Tool 5: How To Conduct a Postdischarge Followup Phone Call Previous Page Next Page Table of Contents (https://www.ahrq.gov) Purpose of This Tool The Re-Engineered Discharge (RED) aims to effectively prepare patients and families for discharge from the hospital, im

## Chunk the Source Documents

In [ ]:
# Splitting each page into smaller overlapping chunks
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
document_chunks = []

for _, row in documents_df.iterrows():
    chunks = chunk_text(
        row["text"]
    )

    for chunk_number, chunk in enumerate(
        chunks,
        start=1
    ):
        document_chunks.append({
            "source": row["source"],
            "title": row["title"],
            "page": row["page"],
            "chunk_number": chunk_number,
            "text": chunk
        })

chunks_df = pd.DataFrame(
    document_chunks
)

print(
    "Document chunks created:",
    len(chunks_df)
)

display(
    chunks_df[
        [
            "source",
            "title",
            "page",
            "chunk_number",
            "text"
        ]
    ].head()
)

Document chunks created: 88


,source,title,page,chunk_number,text
0,AHRQ,Re-Engineered Discharge (RED) Toolkit,1,1,(https://www.ahrq.gov/) Re-Engineered Discharg...
1,AHRQ,Re-Engineered Discharge (RED) Toolkit,1,2,"erse populations, to replicate the RED. Select..."
2,AHRQ,Re-Engineered Discharge (RED) Toolkit,1,3,Lower Readmissions in Dignity Health Hospitals...
3,AHRQ,Re-Engineered Discharge (RED) Toolkit,1,4,"RED in ten hospitals across the country in, ""H..."
4,AHRQ,Re-Engineered Discharge (RED) Toolkit,1,5,hospitals adapted and implemented RED and the ...


In [ ]:
print(
    chunks_df["text"]
    .str.len()
    .describe()
)

count     88.000000
mean     701.727273
std      208.314844
min        3.000000
25%      799.000000
50%      800.000000
75%      800.000000
max      800.000000
Name: text, dtype: float64


In [ ]:
# Removing very short chunks that do not contain useful guidance
chunks_df = chunks_df[
    chunks_df["text"].str.len() >= 100
].reset_index(drop=True)

print(
    "Chunks kept after removing short text:",
    len(chunks_df)
)

print(
    chunks_df["text"]
    .str.len()
    .describe()
)

Chunks kept after removing short text: 85
count     85.000000
mean     725.211765
std      168.933144
min      118.000000
25%      799.000000
50%      800.000000
75%      800.000000
max      800.000000
Name: text, dtype: float64


## Create Document Embeddings and Vector Store

In [ ]:
# Creating embeddings for the healthcare guidance chunks
chunk_embeddings = embed_texts(
    chunks_df["text"].tolist(),
    EMBEDDING_MODEL
)

print(
    "Chunk embedding matrix shape:",
    chunk_embeddings.shape
)

Embedded 85 of 85 rows
Chunk embedding matrix shape: (85, 1536)


In [ ]:
# Building the vector store for document retrieval
document_index = NearestNeighbors(
    n_neighbors=5,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

document_index.fit(
    chunk_embeddings
)

print("Document vector store ready.")

Document vector store ready.


## Test Document Retrieval

A sample hospital readmission question is embedded and compared with the document vector store. The most similar AHRQ and CMS guidance chunks are returned with their source information.

In [ ]:
def retrieve_guidance(query, top_k=5):
    query_embedding = embed_texts(
        [query],
        EMBEDDING_MODEL
    )

    distances, positions = (
        document_index.kneighbors(
            query_embedding,
            n_neighbors=top_k
        )
    )

    similarities = 1.0 - distances[0]

    results = chunks_df.iloc[
        positions[0]
    ].copy()

    results["similarity"] = similarities
    results["retrieval_rank"] = range(
        1,
        len(results) + 1
    )

    return results

In [ ]:
test_query = (
    "What post-discharge interventions should be considered "
    "for a patient at high risk of hospital readmission?"
)

retrieved_guidance = retrieve_guidance(
    test_query,
    top_k=5
)

display(
    retrieved_guidance[
        [
            "retrieval_rank",
            "source",
            "title",
            "page",
            "similarity",
            "text"
        ]
    ]
)

Embedded 1 of 1 rows


,retrieval_rank,source,title,page,similarity,text
2,1,AHRQ,Re-Engineered Discharge (RED) Toolkit,1,0.624761,Lower Readmissions in Dignity Health Hospitals...
13,2,AHRQ,Re-Engineered Discharge (RED) Toolkit,3,0.619143,Follow Up on Test or Lab Results That Are Pend...
0,3,AHRQ,Re-Engineered Discharge (RED) Toolkit,1,0.606379,(https://www.ahrq.gov/) Re-Engineered Discharg...
7,4,AHRQ,Re-Engineered Discharge (RED) Toolkit,2,0.602055,Step 2: Identify Your Implementation Leadershi...
3,5,AHRQ,Re-Engineered Discharge (RED) Toolkit,1,0.600232,"RED in ten hospitals across the country in, ""H..."


In [ ]:
test_query = (
    "What discharge planning, medication review, follow-up appointments, "
    "patient education, or post-discharge follow-up should be provided "
    "to a patient at high risk of 30-day hospital readmission?"
)

retrieved_guidance = retrieve_guidance(
    test_query,
    top_k=5
)

display(
    retrieved_guidance[
        [
            "retrieval_rank",
            "source",
            "title",
            "page",
            "similarity",
            "text"
        ]
    ]
)

Embedded 1 of 1 rows


,retrieval_rank,source,title,page,similarity,text
13,1,AHRQ,Re-Engineered Discharge (RED) Toolkit,3,0.678484,Follow Up on Test or Lab Results That Are Pend...
14,2,AHRQ,Re-Engineered Discharge (RED) Toolkit,3,0.659828,of a Written Discharge Plan in a Way the Patie...
37,3,AHRQ,How To Conduct a Postdischarge Follow-up Phone...,1,0.655950,https://www.ahrq.gov/patient-safety/settings/h...
15,4,AHRQ,Re-Engineered Discharge (RED) Toolkit,3,0.641229,tient- safety/settings/hospital/red/toolkit/re...
36,5,AHRQ,How To Conduct a Postdischarge Follow-up Phone...,1,0.628219,(https://www.ahrq.gov/) Re-Engineered Discharg...


## Generate Evidence-Based Recommendations

The retrieved AHRQ and CMS guidance is provided to an OpenAI language model to generate concise post-discharge recommendations. The model is instructed to use only the retrieved evidence and to cite the source title and page number.

In [ ]:
SYSTEM_INSTRUCTIONS = """
You are a hospital readmission decision-support assistant.

Use only the retrieved AHRQ and CMS guidance provided in the prompt.

Provide concise, practical post-discharge recommendations for patients who may be at risk of 30-day hospital readmission.

Requirements:
- Base every recommendation only on the retrieved evidence.
- Do not add recommendations that are not supported by the retrieved guidance.
- Include the source title and page number for every recommendation.
- Focus on actionable discharge planning, follow-up, medication management, patient education, and care coordination.
- Do not diagnose the patient or replace clinical judgment.
- If the retrieved evidence does not support a recommendation, do not include it.
- Do not offer additional help or ask follow-up questions.
- End after the recommendations and source references.
""".strip()

In [ ]:
def build_rag_prompt(query, retrieved_chunks):
    evidence = []

    for _, row in retrieved_chunks.iterrows():
        evidence.append(
            f"Source: {row['title']} | Page: {row['page']}\n"
            f"{row['text']}"
        )

    evidence_text = "\n\n".join(evidence)

    prompt = f"""
Question:
{query}

Retrieved guidance:
{evidence_text}

Using only the retrieved guidance, provide 3 to 5 practical post-discharge recommendations.

For each recommendation:
1. State the recommended action.
2. Briefly explain why it may help.
3. Cite the source title and page number.

Do not include information that is not supported by the retrieved evidence.
"""

    return prompt

In [ ]:
def generate_rag_recommendation(query, top_k=5):
    retrieved = retrieve_guidance(
        query,
        top_k=top_k
    )

    prompt = build_rag_prompt(
        query,
        retrieved
    )

    response = client.responses.create(
        model=CLASSIFICATION_MODEL,
        instructions=SYSTEM_INSTRUCTIONS,
        input=prompt,
        reasoning={
            "effort": "minimal"
        },
        max_output_tokens=800,
        store=False,
    )

    answer = response.output_text.strip()

    if not answer:
        print("No text response was returned.")
        print("Response status:", response.status)

    return answer, retrieved

In [ ]:
rag_query = (
    "What discharge planning and post-discharge support should be considered "
    "for a patient at high risk of 30-day hospital readmission?"
)

rag_answer, rag_sources = generate_rag_recommendation(
    rag_query,
    top_k=5
)

print(rag_answer)

Embedded 1 of 1 rows
1) Ensure the patient and caregiver receive a written discharge plan in plain language before leaving the hospital.
- Why: A written plan that the patient can understand supports clear expectations, enhances understanding, and reduces likelihood of post-discharge problems.
- Source: Re-Engineered Discharge (RED) Toolkit | Page: 3; Components include "a Written Discharge Plan in a Way the Patient Can Understand" and "Teach the Content of a Written Discharge Plan in a Way the Patient Can Understand" (redtool3a.html#Component6 and redtool3a.html#Com).

2) Conduct and document a postdischarge follow-up phone call within 2–3 days of discharge to review health status, medications, appointments, home services, and steps if problems arise.
- Why: Early follow-up identifies misunderstandings, discrepancies in the discharge plan, and patient concerns, enabling timely resolution and preventing readmission.
- Source: How To Conduct a Postdischarge Follow-up Phone Call | Page: 

In [ ]:
test_queries = [
    "What follow-up support should be provided after discharge to help reduce readmission risk?",
    "What medication and care coordination steps should be considered before and after discharge?"
]

for query in test_queries:
    print("\nQUESTION:")
    print(query)

    answer, sources = generate_rag_recommendation(
        query,
        top_k=5
    )

    print("\nRAG RESPONSE:")
    print(answer)

    print("\n" + "-" * 80)


QUESTION:
What follow-up support should be provided after discharge to help reduce readmission risk?
Embedded 1 of 1 rows

RAG RESPONSE:
Recommendation 1
- Action: Conduct a postdischarge follow-up phone call with all high-risk patients 2 to 3 days after discharge, using a clinical staff member to review health status, medicines, appointments, home services, and contingency plans.
- Why it helps: This call identifies and resolves discrepancies in the discharge plan, addresses patient/caregiver questions and misunderstandings, and ensures planned actions are in place to prevent deterioration and readmission.
- Source: How To Conduct a Postdischarge Follow-up Phone Call, Page 1

Recommendation 2
- Action: Use the follow-up call to verify and address medicines, including reconciliation and ensuring patient understanding of how to obtain and take medications.
- Why it helps: Medication-related issues and misunderstandings are common drivers of readmission; resolving these during the call 

## RAG Evaluation and Limitations

The RAG system successfully retrieved relevant guidance from the AHRQ and CMS source documents for multiple hospital discharge and readmission-related questions. The evidence that was retrieved was used to generate practical recommendations related to follow-up calls, medication management, patient education, care coordination, and discharge planning.

The responses it gave were generally consistent with the retrieved source material and included source titles and page references.

### Limitations

- The system only currently contains a small number of AHRQ and CMS documents.
- The retrieval quality depends on how the user words the question.
- Some of the retrieved chunks may contain overlapping or general guidance rather than patient-specific recommendations.
- The generated recommendations should support, rather than replace, clinical judgment.
- The current RAG system does not independently verify whether every generated statement is clinically appropriate for an individual patient.

In [ ]:
chunks_df.to_csv(
    "rag_document_chunks.csv",
    index=False
)

print("RAG document chunks saved.")

RAG document chunks saved.
